In [1]:
! pip install langchain_community langchainhub chromadb langchain langchain_ollama
! pip install unstructured[epub] unstructured-inference pillow pdf2image
! pip install pypandoc
! python -c "import pypandoc; pypandoc.download_pandoc()"

In [2]:
! curl -fsSL https://ollama.com/install.sh | sh

! ollama pull phi3:mini
! ollama pull nomic-embed-text

! ollama pull llama3:8b-instruct
! ollama pull mxbai-embed-large

>>> Cleaning up old version at /usr/local/lib/ollama
[sudo] password for capti: 
sudo: a password is required
^C
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 633fc5be925f: 100% ▕██████████████████▏ 2.2 GB                         
pulling fa8235e5b48f: 100% ▕██████████████████▏ 1.1 KB                         
pulling 542b217f179c: 100% ▕██████████████████▏  148 B                         
pulling 8dde1baf1db0: 100% ▕██████████████████▏   78 B                         
pulling 23291dc44752: 100% ▕██████████████████▏  483 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕█████████████████

## LOADING DATA

In [4]:
from langchain_community.document_loaders.epub import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


loader = UnstructuredEPubLoader("./archive/pg77221.epub", mode="single")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 50, separators=["\n\n", "\n", ".", "!", "?"])
chunks = splitter.split_documents(documents)
print("Chunks =", len(chunks))

Chunks = 605


## INDEXING

In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

# embedder = OllamaEmbeddings(model = "nomic-embed-text")
embedder = OllamaEmbeddings(model = "mxbai-embed-large")

out = embedder.embed_query("how are you?")
print("Embedding Length =", len(out))


vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    # persist_directory="./indexes/pg77221_chroma_nomic"
    persist_directory="./indexes/pg77221_chroma_mxbai"
)

retriever = vectordb.as_retriever(k=5)

Embedding Length = 1024


## RETREIVAL

In [16]:
from langchain_ollama import ChatOllama
from langchain_classic import hub
# from langchain_core.prompts import ChatPromptTemplate
RAG_PROMPT = hub.pull("rlm/rag-prompt")

llm = ChatOllama(model = "phi3:mini", temperatue = 0, num_ctx=4000)

sample_prompt = RAG_PROMPT.format(context="This is a test context.", question="What is it?")
sample_answer = llm.invoke(sample_prompt)

print(sample_prompt)
print(sample_answer)

Human: You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: What is it? 
Context: This is a test context. 
Answer:
content="It appears to be text from an educational platform or software testing tool designed for question-answering exercises, where 'This' likely refers to either content within the system or instructions provided by users like you asking questions and seeking information on specific topics using available resources such as documents. However, without additional details in this context about a subject matter, I cannot provide an answer to what it is specifically referring to beyond its general purpose of facilitating knowledge acquisition through user queries and accessible content within the system." additional_kwargs={} response_metadata={'model': 'phi3:mini', 'created_at': '20

## GENERATION

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)
c = rag_chain.invoke("what is the name of the book")
print(c)
a = rag_chain.invoke("who are the main characters in this book, name them")
print(a)
b = rag_chain.invoke("what is the most interesting adventure of the main characters")
print(b)

The book is called "Two girls and a mystery." It was originally published by Cupples & Leon Company on November 11, 1928. The current Project Gutenberg version doesn't provide the title directly but refers to it in context with another work named "The old house in the glen," suggesting this may be an alternate or lesser-known edition of a similar mystery story for young readers by May Hollis Barton, illustrated by Ernest N. Townsend.
The main characters in the book are Barbara Winters, Gerry, and a character named Gordon. Additionally, there's also a mention of Rosa Lee, Bab Opens a Door (who seems to be another person), Pattering Feet, an individual referred to as The Hindu, and Sapajou who appears in the storyline.
The most interesting adventure for Barbara among these stories appears to be "Adventure!" as she felt equipped with courage despite overwhelming odds, suggesting an exciting challenge. This aligns closely with the overall themes of thrilling escapades in many tales featuri